In [ ]:
# ==============================
# CELL 1 - IMPORT, UPLOAD, LOAD DATA
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from utils import to_dataframe, FinancialPayload

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

with open('resource/data.json', 'r') as file:
    # Load the JSON data into a Python variable
    data = json.load(file)
    payload_object = FinancialPayload(**data)

df_raw = to_dataframe(payload_object, 'df_joined')
df_raw.head()



           id                   item_name    price category  \
0    trx-0001  akurat strip uji kehamilan    16800   OTHERS   
1    trx-0002                     Sarapan    24200    NEEDS   
2    trx-0003                 GoJek pergi     5800    NEEDS   
3    trx-0004                Gaji Bulanan  9122291    NEEDS   
4    trx-0005                 Bayar Kosan  2032620    NEEDS   
..        ...                         ...      ...      ...   
128  trx-0129                     Sarapan    24200    NEEDS   
129  trx-0130                 GoJek pergi     5800    NEEDS   
130  trx-0131                 Makan Siang    24200    NEEDS   
131  trx-0132                GoJek pulang     5800    NEEDS   
132  trx-0133                 Makan Malam    24200    NEEDS   

                  subcategory  
0         Lain-lain & Darurat  
1        Makan & Minum Harian  
2    Transportasi & Rutinitas  
3                      Salary  
4         Tagihan & Kewajiban  
..                        ...  
128      Makan & Mi

,title,amount,macro_category,master_category,type,timestamp
0,akurat strip uji kehamilan,16800,OTHERS,Lain-lain & Darurat,EXPENSE,2026-05-01T00:44:00Z
1,Sarapan,24200,NEEDS,Makan & Minum Harian,EXPENSE,2026-05-01T07:00:00Z
2,GoJek pergi,5800,NEEDS,Transportasi & Rutinitas,EXPENSE,2026-05-01T08:00:00Z
3,Gaji Bulanan,9122291,NEEDS,Salary,INCOME,2026-05-01T09:00:00Z
4,Bayar Kosan,2032620,NEEDS,Tagihan & Kewajiban,EXPENSE,2026-05-01T10:00:00Z


In [10]:
# ==============================
# CELL 2 - STANDARDIZE & CLEANING
# ==============================

def standardize_mock_finance_columns(df):
    """
    Mapping kolom mock_finance_df_leak.csv ke format API:
    timestamp       -> transaction_date
    amount          -> total_amount
    macro_category  -> category
    master_category -> subcategory
    title           -> description
    """

    df = df.copy()

    rename_map = {
        "timestamp": "transaction_date",
        "amount": "total_amount",
        "macro_category": "category",
        "master_category": "subcategory",
        "title": "description"
    }

    df = df.rename(columns=rename_map)

    required_cols = [
        "transaction_date",
        "type",
        "total_amount",
        "category",
        "subcategory",
        "description"
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

    return df


def clean_transaction_df(df):
    df = df.copy()

    df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
    df["total_amount"] = pd.to_numeric(df["total_amount"], errors="coerce")

    df = df.dropna(subset=[
        "transaction_date",
        "type",
        "total_amount",
        "category",
        "subcategory",
        "description"
    ])

    df["type"] = df["type"].astype(str).str.upper()
    df["category"] = df["category"].astype(str).str.upper()
    df["subcategory"] = df["subcategory"].astype(str)
    df["description"] = df["description"].astype(str)

    df["total_amount"] = df["total_amount"].abs()
    df["date"] = df["transaction_date"].dt.normalize()

    return df


df_standardized = standardize_mock_finance_columns(df_raw)
df_clean = clean_transaction_df(df_standardized)

df_expense = df_clean[df_clean["type"] == "EXPENSE"].copy()
df_income = df_clean[df_clean["type"] == "INCOME"].copy()

print("Shape setelah cleaning:", df_clean.shape)
print("Expense:", df_expense.shape)
print("Income:", df_income.shape)

print("\nDistribusi category:")
display(df_expense["category"].value_counts())

print("\nDistribusi subcategory:")
display(df_expense["subcategory"].value_counts())

display(df_clean.head())

Shape setelah cleaning: (133, 7)
Expense: (132, 7)
Income: (1, 7)

Distribusi category:


category
NEEDS     96
WANTS     22
OTHERS    14
Name: count, dtype: int64


Distribusi subcategory:


subcategory
Makan & Minum Harian        54
Transportasi & Rutinitas    36
Jajan & Nongkrong           16
Lain-lain & Darurat         14
Kebutuhan Rumah & Mandi     10
Tagihan & Kewajiban          2
Name: count, dtype: int64

,description,total_amount,category,subcategory,type,transaction_date,date
0,akurat strip uji kehamilan,16800,OTHERS,Lain-lain & Darurat,EXPENSE,2026-05-01 00:44:00+00:00,2026-05-01 00:00:00+00:00
1,Sarapan,24200,NEEDS,Makan & Minum Harian,EXPENSE,2026-05-01 07:00:00+00:00,2026-05-01 00:00:00+00:00
2,GoJek pergi,5800,NEEDS,Transportasi & Rutinitas,EXPENSE,2026-05-01 08:00:00+00:00,2026-05-01 00:00:00+00:00
3,Gaji Bulanan,9122291,NEEDS,Salary,INCOME,2026-05-01 09:00:00+00:00,2026-05-01 00:00:00+00:00
4,Bayar Kosan,2032620,NEEDS,Tagihan & Kewajiban,EXPENSE,2026-05-01 10:00:00+00:00,2026-05-01 00:00:00+00:00


In [11]:
# ==============================
# CELL 3 - BUILD LEAK CANDIDATE + RULE-BASED LEAK
# ==============================

small_transaction_threshold = 50000
min_frequency = 3
min_total_spending = 100000

habit_keywords = [
    "kopi", "coffee", "rokok", "snack", "jajan",
    "boba", "minuman", "es teh", "cafe", "nongkrong"
]

lifestyle_keywords = [
    "hobi", "self", "reward", "main", "game",
    "top up", "topup", "hiburan", "nongkrong",
    "shopping", "belanja"
]

transport_keywords = [
    "gojek", "grab", "ojek", "transport",
    "angkot", "bus", "kereta"
]


def build_leak_candidate_table(df):
    df_expense_local = df[df["type"] == "EXPENSE"].copy()

    df_small = df_expense_local[
        df_expense_local["total_amount"] <= small_transaction_threshold
    ].copy()

    df_small["description_text"] = df_small["description"].astype(str).str.lower()
    df_small["subcategory_text"] = df_small["subcategory"].astype(str).str.lower()
    df_small["category_text"] = df_small["category"].astype(str).str.lower()

    df_small["is_habit_keyword"] = df_small["description_text"].apply(
        lambda x: any(keyword in x for keyword in habit_keywords)
    )

    df_small["is_lifestyle_keyword"] = (
        df_small["description_text"].apply(
            lambda x: any(keyword in x for keyword in lifestyle_keywords)
        ) |
        df_small["subcategory_text"].apply(
            lambda x: any(keyword in x for keyword in lifestyle_keywords)
        )
    )

    df_small["is_transport_keyword"] = (
        df_small["description_text"].apply(
            lambda x: any(keyword in x for keyword in transport_keywords)
        ) |
        df_small["subcategory_text"].apply(
            lambda x: any(keyword in x for keyword in transport_keywords)
        )
    )

    leak_summary = (
        df_small
        .groupby(["description", "category", "subcategory"], as_index=False)
        .agg(
            frequency=("total_amount", "count"),
            total_spending=("total_amount", "sum"),
            avg_spending=("total_amount", "mean"),
            first_date=("date", "min"),
            last_date=("date", "max"),
            habit_keyword_count=("is_habit_keyword", "sum"),
            lifestyle_keyword_count=("is_lifestyle_keyword", "sum"),
            transport_keyword_count=("is_transport_keyword", "sum")
        )
    )

    leak_summary["active_days"] = (
        leak_summary["last_date"] - leak_summary["first_date"]
    ).dt.days + 1

    leak_summary["avg_days_between_purchase"] = (
        leak_summary["active_days"] / leak_summary["frequency"]
    )

    leak_summary["is_repeated_small_spending"] = (
        (leak_summary["frequency"] >= min_frequency) &
        (leak_summary["total_spending"] >= min_total_spending)
    ).astype(int)

    leak_summary["category_text"] = leak_summary["category"].astype(str).str.lower()

    return leak_summary, df_small


def classify_rule_based_leak(row):
    is_repeated = row["is_repeated_small_spending"] == 1

    is_needs = row["category_text"] == "needs"
    is_wants = row["category_text"] == "wants"
    is_others = row["category_text"] == "others"

    is_habit = row["habit_keyword_count"] > 0
    is_lifestyle = row["lifestyle_keyword_count"] > 0
    is_transport = row["transport_keyword_count"] > 0

    if not is_repeated:
        return "Normal"

    if is_transport and is_needs:
        return "Recurring Essential"

    if is_transport and is_wants:
        return "Lifestyle Leak"

    if is_needs:
        return "Recurring Essential"

    if is_habit:
        return "Habit Leak"

    if is_lifestyle or is_wants or is_others:
        return "Lifestyle Leak"

    return "Potential Leak"


leak_summary, df_small = build_leak_candidate_table(df_clean)

leak_summary["rule_leak_type"] = leak_summary.apply(classify_rule_based_leak, axis=1)

leak_summary["is_rule_potential_leak"] = leak_summary["rule_leak_type"].isin([
    "Habit Leak",
    "Lifestyle Leak",
    "Potential Leak"
]).astype(int)

print("Jumlah transaksi kecil:", len(df_small))
print("Jumlah item kandidat leak:", len(leak_summary))
print("Rule potential leak:", leak_summary["is_rule_potential_leak"].sum())

display(leak_summary.sort_values("total_spending", ascending=False).head(20))

Jumlah transaksi kecil: 125
Jumlah item kandidat leak: 29
Rule potential leak: 0


,description,category,subcategory,frequency,total_spending,avg_spending,first_date,last_date,habit_keyword_count,lifestyle_keyword_count,transport_keyword_count,active_days,avg_days_between_purchase,is_repeated_small_spending,category_text,rule_leak_type,is_rule_potential_leak
4,Makan Siang,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0
3,Makan Malam,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0
5,Sarapan,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0
2,GoJek pulang,NEEDS,Transportasi & Rutinitas,18,104400,5800.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,18,18,1.000000,1,needs,Recurring Essential,0
0,GoJek pergi,NEEDS,Transportasi & Rutinitas,12,69600,5800.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,12,18,1.500000,0,needs,Normal,0
28,yupi wild safari permen jeli aneka rasa 44 g,WANTS,Jajan & Nongkrong,13,63700,4900.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,13,0,18,1.384615,0,wants,Normal,0
22,vicks formula 44 obat batuk cair 100 ml,OTHERS,Lain-lain & Darurat,1,40200,40200.0,2026-05-09 00:00:00+00:00,2026-05-09 00:00:00+00:00,0,0,0,1,1.000000,0,others,Normal,0
24,whiskas makanan kucing kering adult tuna 480 g,OTHERS,Lain-lain & Darurat,1,38200,38200.0,2026-05-11 00:00:00+00:00,2026-05-11 00:00:00+00:00,0,0,0,1,1.000000,0,others,Normal,0
1,GoJek pergi,WANTS,Transportasi & Rutinitas,6,34800,5800.0,2026-05-02 00:00:00+00:00,2026-05-17 00:00:00+00:00,0,0,6,16,2.666667,0,wants,Normal,0
25,whiskas makanan kucing kering kitten makerel 4...,OTHERS,Lain-lain & Darurat,1,31900,31900.0,2026-05-02 00:00:00+00:00,2026-05-02 00:00:00+00:00,0,0,0,1,1.000000,0,others,Normal,0


In [12]:
# ==============================
# CELL 4 - LEAK SCORE + TRAIN ML ANOMALY
# ==============================

score_feature_columns = [
    "frequency",
    "total_spending",
    "avg_spending",
    "habit_keyword_count",
    "lifestyle_keyword_count"
]

score_features = leak_summary[score_feature_columns].copy()
score_features = score_features.replace([np.inf, -np.inf], np.nan).fillna(0)

for col in ["frequency", "total_spending", "avg_spending"]:
    score_features[col] = np.log1p(score_features[col])

leak_score_scaler = MinMaxScaler()
scaled_score_features = leak_score_scaler.fit_transform(score_features)

leak_score_weights = np.array([0.30, 0.35, 0.10, 0.15, 0.10])

leak_score_raw = scaled_score_features @ leak_score_weights

leak_summary["leak_score"] = leak_score_raw * 100
leak_summary.loc[leak_summary["category_text"] == "needs", "leak_score"] *= 0.65
leak_summary["leak_score"] = leak_summary["leak_score"].clip(0, 100).round(2)

# ML anomaly features
leak_summary["is_needs"] = (leak_summary["category_text"] == "needs").astype(int)
leak_summary["is_wants"] = (leak_summary["category_text"] == "wants").astype(int)
leak_summary["is_others"] = (leak_summary["category_text"] == "others").astype(int)

ml_features = [
    "frequency",
    "total_spending",
    "avg_spending",
    "active_days",
    "avg_days_between_purchase",
    "habit_keyword_count",
    "lifestyle_keyword_count",
    "transport_keyword_count",
    "leak_score",
    "is_needs",
    "is_wants",
    "is_others"
]

X = leak_summary[ml_features].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

log_cols = [
    "frequency",
    "total_spending",
    "avg_spending",
    "active_days",
    "avg_days_between_purchase"
]

for col in log_cols:
    X[col] = np.log1p(X[col])

ml_scaler = RobustScaler()
X_scaled = ml_scaler.fit_transform(X)

isolation_model = IsolationForest(
    n_estimators=300,
    contamination=0.10,
    random_state=42
)

leak_summary["ml_anomaly_label"] = isolation_model.fit_predict(X_scaled)
leak_summary["is_ml_anomaly"] = (leak_summary["ml_anomaly_label"] == -1).astype(int)

raw_score = -isolation_model.decision_function(X_scaled)

training_score_min = raw_score.min()
training_score_max = raw_score.max()

if training_score_max - training_score_min == 0:
    leak_summary["ml_anomaly_score"] = 0
else:
    leak_summary["ml_anomaly_score"] = (
        (raw_score - training_score_min) /
        (training_score_max - training_score_min) * 100
    )

leak_summary["ml_anomaly_score"] = leak_summary["ml_anomaly_score"].clip(0, 100).round(2)

leak_summary["is_financially_relevant_anomaly"] = (
    (leak_summary["frequency"] >= min_frequency) |
    (leak_summary["total_spending"] >= min_total_spending) |
    (leak_summary["leak_score"] >= 40)
).astype(int)


def classify_ml_leak(row):
    is_anomaly = row["is_ml_anomaly"] == 1
    is_relevant = row["is_financially_relevant_anomaly"] == 1

    is_needs = row["category_text"] == "needs"
    is_wants = row["category_text"] == "wants"
    is_others = row["category_text"] == "others"
    is_transport = row["transport_keyword_count"] > 0

    if not is_anomaly:
        return "Normal"

    if not is_relevant:
        return "ML Minor Anomaly"

    if is_needs and is_transport:
        return "ML Recurring Essential Anomaly"

    if is_needs:
        return "ML Needs Anomaly"

    if is_wants:
        return "ML Lifestyle Leak Anomaly"

    if is_others:
        return "ML Other Leak Anomaly"

    return "ML Potential Leak Anomaly"


leak_summary["ml_leak_type"] = leak_summary.apply(classify_ml_leak, axis=1)

leak_summary["is_ml_potential_leak"] = (
    (leak_summary["is_ml_anomaly"] == 1) &
    (leak_summary["is_financially_relevant_anomaly"] == 1) &
    (leak_summary["category_text"].isin(["wants", "others"]))
).astype(int)

leak_summary["is_final_potential_leak"] = (
    (leak_summary["is_rule_potential_leak"] == 1) |
    (leak_summary["is_ml_potential_leak"] == 1)
).astype(int)

leak_summary = leak_summary.sort_values(
    by=[
        "is_final_potential_leak",
        "is_ml_anomaly",
        "ml_anomaly_score",
        "leak_score",
        "total_spending"
    ],
    ascending=False
).reset_index(drop=True)

print("===== LEAK DETECTION RESULT =====")
print("ML anomaly:", leak_summary["is_ml_anomaly"].sum())
print("ML potential leak:", leak_summary["is_ml_potential_leak"].sum())
print("Final potential leak:", leak_summary["is_final_potential_leak"].sum())

display(leak_summary.head(20))

===== LEAK DETECTION RESULT =====
ML anomaly: 3
ML potential leak: 2
Final potential leak: 2


,description,category,subcategory,frequency,total_spending,avg_spending,first_date,last_date,habit_keyword_count,lifestyle_keyword_count,transport_keyword_count,active_days,avg_days_between_purchase,is_repeated_small_spending,category_text,rule_leak_type,is_rule_potential_leak,leak_score,is_needs,is_wants,is_others,ml_anomaly_label,is_ml_anomaly,ml_anomaly_score,is_financially_relevant_anomaly,ml_leak_type,is_ml_potential_leak,is_final_potential_leak
0,yupi wild safari permen jeli aneka rasa 44 g,WANTS,Jajan & Nongkrong,13,63700,4900.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,13,0,18,1.384615,0,wants,Normal,0,54.10,0,1,0,-1,1,100.00,1,ML Lifestyle Leak Anomaly,1,1
1,GoJek pergi,WANTS,Transportasi & Rutinitas,6,34800,5800.0,2026-05-02 00:00:00+00:00,2026-05-17 00:00:00+00:00,0,0,6,16,2.666667,0,wants,Normal,0,30.37,0,1,0,-1,1,90.14,1,ML Lifestyle Leak Anomaly,1,1
2,kopi toraja,WANTS,Jajan & Nongkrong,1,20000,20000.0,2026-05-06 00:00:00+00:00,2026-05-06 00:00:00+00:00,1,1,0,1,1.000000,0,wants,Normal,0,30.47,0,1,0,-1,1,85.83,0,ML Minor Anomaly,0,0
3,GoJek pulang,NEEDS,Transportasi & Rutinitas,18,104400,5800.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,18,18,1.000000,1,needs,Recurring Essential,0,34.64,1,0,0,1,0,72.11,1,Normal,0,0
4,GoJek pergi,NEEDS,Transportasi & Rutinitas,12,69600,5800.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,12,18,1.500000,0,needs,Normal,0,29.05,1,0,0,1,0,72.01,1,Normal,0,0
5,lotte xylitol fresh mint 26.1 g,WANTS,Jajan & Nongkrong,1,9600,9600.0,2026-05-11 00:00:00+00:00,2026-05-11 00:00:00+00:00,0,1,0,1,1.000000,0,wants,Normal,0,5.56,0,1,0,1,0,55.00,0,Normal,0,0
6,Makan Malam,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0,47.18,1,0,0,1,0,49.96,1,Normal,0,0
7,Makan Siang,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0,47.18,1,0,0,1,0,49.96,1,Normal,0,0
8,Sarapan,NEEDS,Makan & Minum Harian,18,435600,24200.0,2026-05-01 00:00:00+00:00,2026-05-18 00:00:00+00:00,0,0,0,18,1.000000,1,needs,Recurring Essential,0,47.18,1,0,0,1,0,49.96,1,Normal,0,0
9,vicks formula 44 obat batuk cair 100 ml,OTHERS,Lain-lain & Darurat,1,40200,40200.0,2026-05-09 00:00:00+00:00,2026-05-09 00:00:00+00:00,0,0,0,1,1.000000,0,others,Normal,0,24.14,0,0,1,1,0,28.12,0,Normal,0,0


In [13]:
# ==============================
# CELL 5 - FINANCIAL SCORE
# ==============================

def calculate_financial_score_from_dataframe(df, leak_summary=None, budgets=None):
    df = df.copy()

    df["type"] = df["type"].astype(str).str.upper()
    df["category"] = df["category"].astype(str).str.upper()
    df["total_amount"] = pd.to_numeric(df["total_amount"], errors="coerce").fillna(0)

    df_expense_local = df[df["type"] == "EXPENSE"].copy()
    df_income_local = df[df["type"] == "INCOME"].copy()

    total_income = df_income_local["total_amount"].sum()
    total_expense = df_expense_local["total_amount"].sum()
    net_cashflow = total_income - total_expense

    savings_rate = net_cashflow / total_income if total_income > 0 else 0

    category_expense = (
        df_expense_local
        .groupby("category", as_index=False)["total_amount"]
        .sum()
    )

    category_dict = dict(zip(category_expense["category"], category_expense["total_amount"]))

    needs_amount = category_dict.get("NEEDS", 0)
    wants_amount = category_dict.get("WANTS", 0)
    others_amount = category_dict.get("OTHERS", 0)

    needs_ratio = needs_amount / total_expense if total_expense > 0 else 0
    wants_ratio = wants_amount / total_expense if total_expense > 0 else 0
    others_ratio = others_amount / total_expense if total_expense > 0 else 0

    daily_expense = (
        df_expense_local
        .groupby("date", as_index=False)["total_amount"]
        .sum()
    )

    daily_mean = daily_expense["total_amount"].mean()
    daily_std = daily_expense["total_amount"].std()

    spending_volatility = daily_std / daily_mean if daily_mean > 0 else 0

    if leak_summary is not None and not leak_summary.empty:
        final_potential_leak_count = int(leak_summary["is_final_potential_leak"].sum())

        final_potential_leak_spending = leak_summary.loc[
            leak_summary["is_final_potential_leak"] == 1,
            "total_spending"
        ].sum()
    else:
        final_potential_leak_count = 0
        final_potential_leak_spending = 0

    potential_leak_ratio = (
        final_potential_leak_spending / total_expense if total_expense > 0 else 0
    )

    budget_used_ratio_total = 0
    overbudget_category_count = 0

    if budgets is not None and len(budgets) > 0:
        budget_df = pd.DataFrame(budgets)

        budget_df["category"] = budget_df["category"].astype(str).str.upper()
        budget_df["limit_amount"] = pd.to_numeric(
            budget_df["limit_amount"], errors="coerce"
        ).fillna(0)

        expense_by_category = (
            df_expense_local
            .groupby("category", as_index=False)["total_amount"]
            .sum()
            .rename(columns={"total_amount": "actual_spending"})
        )

        budget_check = budget_df.merge(
            expense_by_category,
            on="category",
            how="left"
        )

        budget_check["actual_spending"] = budget_check["actual_spending"].fillna(0)

        budget_check["budget_used_ratio"] = np.where(
            budget_check["limit_amount"] > 0,
            budget_check["actual_spending"] / budget_check["limit_amount"],
            0
        )

        total_budget = budget_check["limit_amount"].sum()
        total_budget_spending = budget_check["actual_spending"].sum()

        budget_used_ratio_total = (
            total_budget_spending / total_budget if total_budget > 0 else 0
        )

        overbudget_category_count = int(
            (budget_check["budget_used_ratio"] > 1).sum()
        )

    financial_score = 100
    score_reasons = []

    if net_cashflow < 0:
        financial_score -= 25
        score_reasons.append("Cashflow negatif karena pengeluaran lebih besar dari pemasukan.")
    elif savings_rate < 0.10:
        financial_score -= 8
        score_reasons.append("Savings rate masih rendah.")

    if wants_ratio > 0.35:
        financial_score -= 15
        score_reasons.append("Proporsi pengeluaran WANTS cukup tinggi.")
    elif wants_ratio > 0.25:
        financial_score -= 8
        score_reasons.append("Proporsi pengeluaran WANTS perlu dipantau.")

    if others_ratio > 0.15:
        financial_score -= 10
        score_reasons.append("Proporsi pengeluaran OTHERS cukup tinggi.")

    if final_potential_leak_count >= 5:
        financial_score -= 20
        score_reasons.append("Terdapat banyak potensi leak.")
    elif final_potential_leak_count >= 1:
        financial_score -= 10
        score_reasons.append("Terdapat beberapa potensi leak.")

    if potential_leak_ratio > 0.15:
        financial_score -= 10
        score_reasons.append("Total leak cukup besar dibanding total pengeluaran.")

    if budget_used_ratio_total > 1:
        financial_score -= 15
        score_reasons.append("Total pengeluaran melewati budget.")
    elif budget_used_ratio_total > 0.85:
        financial_score -= 8
        score_reasons.append("Pemakaian budget sudah mendekati batas.")

    if spending_volatility > 1:
        financial_score -= 10
        score_reasons.append("Pengeluaran harian cukup fluktuatif.")

    financial_score = max(0, min(100, financial_score))

    if financial_score >= 80:
        score_category = "Excellent"
    elif financial_score >= 65:
        score_category = "Good"
    elif financial_score >= 50:
        score_category = "Fair"
    else:
        score_category = "Needs Attention"

    if len(score_reasons) == 0:
        score_reasons.append("Kondisi keuangan relatif stabil dan tidak banyak risiko terdeteksi.")

    return {
        "financial_score": int(financial_score),
        "score_category": score_category,
        "total_income": float(total_income),
        "total_expense": float(total_expense),
        "net_cashflow": float(net_cashflow),
        "savings_rate": float(savings_rate),
        "needs_ratio": float(needs_ratio),
        "wants_ratio": float(wants_ratio),
        "others_ratio": float(others_ratio),
        "final_potential_leak_count": int(final_potential_leak_count),
        "final_potential_leak_spending": float(final_potential_leak_spending),
        "potential_leak_ratio": float(potential_leak_ratio),
        "budget_used_ratio_total": float(budget_used_ratio_total),
        "overbudget_category_count": int(overbudget_category_count),
        "spending_volatility": float(spending_volatility),
        "score_reason": " ".join(score_reasons)
    }


financial_score_result = calculate_financial_score_from_dataframe(
    df_clean,
    leak_summary=leak_summary,
    budgets=None
)

display(pd.DataFrame([financial_score_result]))

,financial_score,score_category,total_income,total_expense,net_cashflow,savings_rate,needs_ratio,wants_ratio,others_ratio,final_potential_leak_count,final_potential_leak_spending,potential_leak_ratio,budget_used_ratio_total,overbudget_category_count,spending_volatility,score_reason
0,80,Excellent,9122291.0,4806967.0,4315324.0,0.473053,0.890263,0.040171,0.069566,2,98500.0,0.020491,0.0,0,2.204334,Terdapat beberapa potensi leak. Pengeluaran ha...


In [15]:
financial_score_result

{'financial_score': 80,
 'score_category': 'Excellent',
 'total_income': 9122291.0,
 'total_expense': 4806967.0,
 'net_cashflow': 4315324.0,
 'savings_rate': 0.47305265749579795,
 'needs_ratio': 0.8902634447043218,
 'wants_ratio': 0.04017086033667383,
 'others_ratio': 0.06956569495900429,
 'final_potential_leak_count': 2,
 'final_potential_leak_spending': 98500.0,
 'potential_leak_ratio': 0.020491091368008144,
 'budget_used_ratio_total': 0.0,
 'overbudget_category_count': 0,
 'spending_volatility': 2.2043340878795705,
 'score_reason': 'Terdapat beberapa potensi leak. Pengeluaran harian cukup fluktuatif.'}

In [14]:
category_summary = (
    df_expense
    .groupby("category", as_index=False)
    .agg(
        total_spending=("total_amount", "sum"),
        transaction_count=("total_amount", "count"),
        avg_spending=("total_amount", "mean")
    )
    .sort_values("total_spending", ascending=False)
)

display(category_summary)

,category,total_spending,transaction_count,avg_spending
0,NEEDS,4279467,96,44577.781250
1,OTHERS,334400,14,23885.714286
2,WANTS,193100,22,8777.272727


In [22]:
leak_products = leak_summary[leak_summary["is_final_potential_leak"] == 1]
leak_products_list = leak_products['description'].to_list()
leak_products_list

['yupi wild safari permen jeli aneka rasa 44 g', 'GoJek pergi']

In [ ]:
result = {"financial summary":financial_score_result,
          "leak_products": leak_products_list
}